# Sentiment Stability Analysis

This notebook loads the production sentiment stability feature set and yearly market returns, merges them by year, and evaluates whether sentiment deviation is related to forward returns.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == 'notebooks' else cwd

BASE_DIR = get_project_root()
if str(BASE_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(BASE_DIR / 'src'))

STABILITY_PATH = BASE_DIR / 'features' / 'sentiment_stability.csv'
MARKET_PATH = BASE_DIR / 'features' / 'market_returns.csv'

print(f'Loading sentiment stability features from: {STABILITY_PATH}')
print(f'Loading market returns from: {MARKET_PATH}')

stability = pd.read_csv(STABILITY_PATH)
market_returns = pd.read_csv(MARKET_PATH)

stability['year'] = pd.to_numeric(stability['year'], errors='coerce').astype('Int64')
market_returns['year'] = pd.to_numeric(market_returns['year'], errors='coerce').astype('Int64')

stability = stability.sort_values(['company_id', 'year'], kind='mergesort').reset_index(drop=True)
market_returns = market_returns.sort_values('year', kind='mergesort').reset_index(drop=True)

print('stability rows:', len(stability))
print('market rows:', len(market_returns))
print('stability year range:', stability['year'].min(), '->', stability['year'].max())
print('market year range:', market_returns['year'].min(), '->', market_returns['year'].max())


In [ ]:
analysis_df = stability.merge(market_returns, on='year', how='left', validate='many_to_one')
analysis_df = analysis_df.dropna(subset=['sentiment_deviation', 'next_year_return']).copy()
analysis_df['deviation_bucket'] = pd.qcut(analysis_df['sentiment_deviation'], q=3, labels=['low', 'mid', 'high'], duplicates='drop')

print('merged rows:', len(analysis_df))
analysis_df.head()


In [ ]:
correlation = analysis_df['sentiment_deviation'].corr(analysis_df['next_year_return'])

bucket_distribution = (
    analysis_df.groupby('deviation_bucket', observed=False)['next_year_return']
    .agg(
        count='size',
        mean_return='mean',
        median_return='median',
        std_return='std',
        min_return='min',
        max_return='max',
    )
    .sort_index()
)

negative_mask = analysis_df['next_year_return'] < 0
negative_returns = analysis_df['next_year_return'].where(negative_mask)
upside_mask = analysis_df['next_year_return'] > 0.20
bucket_top_quintile = (
    analysis_df.groupby('deviation_bucket', observed=False)['next_year_return']
    .quantile(0.80)
    .reset_index(name='top_quintile_threshold')
)
analysis_with_thresholds = analysis_df.merge(
    bucket_top_quintile,
    on='deviation_bucket',
    how='left',
    validate='many_to_one',
)

risk_profile = (
    analysis_with_thresholds.assign(
        is_negative=negative_mask,
        negative_return=negative_returns,
        is_above_20=upside_mask,
        top_20pct_return=analysis_with_thresholds['next_year_return'].where(
            analysis_with_thresholds['next_year_return'] >= analysis_with_thresholds['top_quintile_threshold']
        ),
    )
    .groupby('deviation_bucket', observed=False)
    .agg(
        negative_return_pct=('is_negative', 'mean'),
        average_negative_return=('negative_return', 'mean'),
        above_20pct_return_pct=('is_above_20', 'mean'),
        average_top_20pct_return=('top_20pct_return', 'mean'),
    )
    .sort_index()
)

summary_table = bucket_distribution.join(risk_profile)
percentage_columns = ['negative_return_pct', 'above_20pct_return_pct']
summary_table[percentage_columns] = summary_table[percentage_columns].mul(100)
summary_table = summary_table.round(4)

print('correlation (deviation vs forward returns):', round(correlation, 4))
display(summary_table)


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    analysis_df['sentiment_deviation'],
    analysis_df['next_year_return'].abs(),
    alpha=0.7,
)
plt.title('Sentiment Deviation vs Absolute Forward Returns')
plt.xlabel('Sentiment Deviation')
plt.ylabel('|Next-Year Return|')
plt.tight_layout()
plt.show()


In [ ]:
yearly_summary = (
    analysis_df.groupby('year', as_index=False)
    .agg(
        average_deviation=('sentiment_deviation', 'mean'),
        average_return=('next_year_return', 'mean'),
    )
    .sort_values('year', kind='mergesort')
)

rolling_window = (
    yearly_summary.assign(
        rolling_average_deviation=yearly_summary['average_deviation'].rolling(window=5, min_periods=5).mean(),
        rolling_average_return=yearly_summary['average_return'].rolling(window=5, min_periods=5).mean(),
    )
    .dropna(subset=['rolling_average_deviation', 'rolling_average_return'])
    .assign(window_end_year=lambda df: df['year'])
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(
    rolling_window['window_end_year'],
    rolling_window['rolling_average_deviation'],
    color='tab:orange',
    marker='o',
    label='5Y Avg Deviation',
)
ax1.set_xlabel('Window End Year')
ax1.set_ylabel('5Y Avg Sentiment Deviation', color='tab:orange')
ax1.tick_params(axis='y', labelcolor='tab:orange')

ax2 = ax1.twinx()
ax2.plot(
    rolling_window['window_end_year'],
    rolling_window['rolling_average_return'],
    color='tab:green',
    marker='s',
    label='5Y Avg Return',
)
ax2.set_ylabel('5Y Avg Next-Year Return', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='best')
plt.title('Rolling 5-Year Sentiment Deviation and Forward Returns')
plt.tight_layout()
plt.show()

rolling_window.tail()
